In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "position"                      
BRONZE_SOURCE_NAME = "external_position"       # Bronze table name
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, BRONZE_SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze row count: 21
root
 |-- position_id: string (nullable = true)
 |-- fund_id: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- quantity: double (nullable = true)
 |-- price: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _source_file_record_count_mismatch: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



### 1. Type casting (defensive - Bronze may already have done this)

In [0]:
typed_df = (
    bronze_df
    .withColumn("position_id", F.trim(F.col("position_id")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))          # external code, e.g. FND001
    .withColumn("asset_id", F.trim(F.col("asset_id")))        # external code, e.g. AST001
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    # Bronze doesn't have a business_date column - derive it from timestamp
    # (the .dat filename encodes the business date, but Bronze only kept the
    # per-record timestamp, so we extract the date part from that instead).
    .withColumn("business_date", F.to_date(F.col("timestamp")))
)

### 2. Crosswalk join - resolve external IDs to internal IDs
A lookup miss here is unexpected for VALID files (FNDXXX/AST999 only
appear in the deliberately-invalid file, which Bronze already
quarantined) - so any miss here gets flagged, not silently dropped.

In [0]:
fund_xwalk = read_crosswalk(spark, "fund")   
asset_xwalk = read_crosswalk(spark, "asset")

# Adjust these column names to match your actual crosswalk table schema

FUND_XWALK_EXTERNAL_COL = "external_fund_id"
FUND_XWALK_INTERNAL_COL = "internal_fund_id"
ASSET_XWALK_EXTERNAL_COL = "external_asset_id"
ASSET_XWALK_INTERNAL_COL = "internal_company_id"   

fund_xwalk_slim = fund_xwalk.select(
    F.col(FUND_XWALK_EXTERNAL_COL).alias("fund_id"),
    F.col(FUND_XWALK_INTERNAL_COL).alias("internal_fund_id"),
)
asset_xwalk_slim = asset_xwalk.select(
    F.col(ASSET_XWALK_EXTERNAL_COL).alias("asset_id"),
    F.col(ASSET_XWALK_INTERNAL_COL).alias("internal_asset_id"),
)

joined_df = (
    typed_df
    .join(fund_xwalk_slim, on="fund_id", how="left")
    .join(asset_xwalk_slim, on="asset_id", how="left")
)

xwalk_miss_df = joined_df.filter(F.col("internal_fund_id").isNull() | F.col("internal_asset_id").isNull()) \
    .withColumn("reason_code", F.lit("UNKNOWN_CROSSWALK_MAPPING"))
xwalk_miss_count = xwalk_miss_df.count()
if xwalk_miss_count > 0:
    write_quarantine(xwalk_miss_df, SOURCE_NAME)
    print(f"WARNING: {xwalk_miss_count} rows failed crosswalk lookup - investigate, this is unexpected for valid files.")

crosswalked_df = joined_df.filter(F.col("internal_fund_id").isNotNull() & F.col("internal_asset_id").isNotNull())

### 3. Duplicate / POSITION_BREAK detection
Business key : internal_fund_id +
internal_asset_id + business_date + quantity + price.
Day-2 FND002/AST004 conflicting quantities (7,300 vs 7,050) should
surface here as a POSITION_BREAK. Day-3 exact repeat
(POS-DUP-20260917-001) should surface as a duplicate.

In [0]:
KEY_COLS = ["internal_fund_id", "internal_asset_id", "business_date"]
COMPARE_COLS = ["quantity", "price"]

deduped_df, duplicates_df, breaks_df = split_duplicates(crosswalked_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()

if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("POSITION_BREAK")), SOURCE_NAME)
    print(f"POSITION_BREAK: {break_count} rows - expected to include the Day-2 FND002/AST004 case.")

POSITION_BREAK: 4 rows - expected to include the Day-2 FND002/AST004 case.


### 4. Write to Silver


In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 20


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "crosswalk_miss", joined_df.count(), xwalk_miss_count, "UNKNOWN_CROSSWALK_MAPPING")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", crosswalked_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "position_break", crosswalked_df.count(), break_count, "POSITION_BREAK")

/home/spark-d25bad90-c236-40c3-82f2-c6/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = xwalk_miss_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=21 = silver=20 + quarantined=1
